# Statistical Concepts & Probability Distributions (CFA Level 1)
## Descriptive Statistics, Moments, and Distribution Theory for Investment Analysis

---

### Why Statistics Matters for Investors

Every investment decision involves uncertainty. You never know *exactly* what return you'll earn, whether a stock will go up or down, or how volatile the market will be next month. Statistics gives us the tools to **quantify this uncertainty** and make informed decisions despite it.

Think of statistics as the language of uncertainty. Just as an accountant needs to read financial statements, an investment professional needs to read and interpret statistical measures. When someone says a portfolio has "12% expected return with 20% standard deviation," you need to understand what that *means* for an investor.

**What you will learn:**
1. Descriptive statistics: measures of central tendency and dispersion
2. Higher moments: skewness and kurtosis of financial returns
3. Key probability distributions (discrete and continuous)
4. The normal distribution and its applications in finance
5. Hypothesis testing from scratch

> **Key Concept:** Statistics is not about numbers for their own sake. Every statistical measure we'll learn has a direct **investment interpretation**. Variance tells you about risk. Skewness tells you about crash probability. The Sharpe ratio tells you about risk-adjusted performance.

**Prerequisites:** Basic algebra, familiarity with summation notation.

**References:**
- CFA Institute, *CFA Program Curriculum*, Quantitative Methods.
- DeFusco, R. et al., *Quantitative Investment Analysis*, CFA Institute, Wiley.

### Why this notebook matters

Statistics is the language of finance. Every claim about "expected return" relies on a population mean. Every "risk measure" relies on standard deviation or variance. Every "statistical significance" claim relies on hypothesis testing. Without these tools, financial analysis would be guesswork.

> **Key Concept:** The CFA curriculum teaches statistics not as an end in itself but as a *toolkit* for financial decisions. The same mean, variance, skewness, and kurtosis concepts appear repeatedly across portfolio theory (efficient frontier), risk management (Value at Risk), and performance evaluation (Sharpe ratio). Mastering these foundations pays dividends across every other CFA topic.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 1. Descriptive Statistics: Measures of Central Tendency

### The Big Question: "What's Typical?"

When looking at a set of investment returns, the first question you naturally ask is: *"What's the average?"* But there are several different kinds of "average," and each tells you something different.

### Arithmetic Mean: The Familiar Average

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

The arithmetic mean is what most people think of as "the average." You add up all the values and divide by how many there are.

**Investment meaning:** The arithmetic mean return is the **best estimate of the return in any single future period**, assuming returns are drawn from the same distribution. If a fund averaged 8% over the past 10 years, 8% is our best guess for next year's return.

**Worked Example:** Annual returns of 15%, -8%, 12%, 6%, -3%.
$$\bar{x} = \frac{0.15 + (-0.08) + 0.12 + 0.06 + (-0.03)}{5} = \frac{0.22}{5} = 4.4\%$$

### Weighted Mean: When Not All Values Are Equal

$$\bar{x}_w = \sum_{i=1}^{n} w_i x_i, \quad \text{where } \sum w_i = 1$$

**Investment meaning:** This is how we calculate **portfolio returns**. If you have 60% in stocks (returning 10%) and 40% in bonds (returning 4%), your portfolio return is $0.60 \times 10\% + 0.40 \times 4\% = 7.6\%$.

### Geometric Mean: The Compound Growth Rate

$$\bar{x}_G = \left(\prod_{i=1}^{n} (1 + x_i)\right)^{1/n} - 1$$

**Investment meaning:** The geometric mean tells you the **actual compound growth rate** of your investment. If you invested $1,000 and earned returns of 20%, -10%, and 15% over three years, the geometric mean tells you the single constant rate that would produce the same ending wealth.

**Why it differs from the arithmetic mean:** The geometric mean is *always* less than or equal to the arithmetic mean (with equality only when all returns are identical). This happens because of **volatility drag** -- the negative effect of fluctuations on compound growth.

**Worked Example:** Returns of +50% and -50% over two years.
- Arithmetic mean: $(50\% + (-50\%))/2 = 0\%$ -- looks break-even!
- But: $1,000 \times 1.50 \times 0.50 = \$750$ -- you actually *lost* money!
- Geometric mean: $(1.50 \times 0.50)^{1/2} - 1 = 0.866 - 1 = -13.4\%$ -- correctly shows the loss.

> **Key Concept:** The arithmetic mean is always the more optimistic measure. The geometric mean captures the reality of compound growth. For evaluating **past performance**, always use the geometric mean. For **forecasting** expected return in a single period, use the arithmetic mean.

### Harmonic Mean: The Dollar-Cost Averager's Friend

$$\bar{x}_H = \frac{n}{\sum_{i=1}^{n} \frac{1}{x_i}}$$

**Investment meaning:** If you invest a fixed dollar amount regularly (dollar-cost averaging), the harmonic mean of the prices is your average cost per share.

**Example:** You invest $100/month when the stock is at $10, $12, $8, $11, $9.
- You buy: 10, 8.33, 12.5, 9.09, 11.11 shares = 51.03 shares total
- Average cost: $500/51.03 = **$9.80** (which equals the harmonic mean)
- Arithmetic mean of prices: $10.00 (higher!)

You automatically buy more shares when prices are low, which lowers your average cost below the arithmetic average.

### The Inequality: $\bar{x}_H \leq \bar{x}_G \leq \bar{x}$

For positive data, the harmonic mean is always the smallest, the arithmetic mean is always the largest, and the geometric mean falls in between. They are all equal only when every data point has the same value.

### Median and Mode

- **Median:** The middle value when data is sorted. Useful because it's **not affected by outliers** -- a single extreme return doesn't distort it.
- **Mode:** The most frequent value. Less commonly used in finance (continuous data rarely has exact repeats).

> **CFA Exam Tip:** Know when to use each mean. The exam loves to test whether you can identify which average is appropriate: arithmetic for single-period forecasts, geometric for multi-period compound returns, harmonic for dollar-cost averaging.

Let's implement all of these and see them in action with real-world-style data.

### Three measures of central tendency

The three classical measures of "what is typical" each have specific use cases:

| Measure | Formula | Best Use Case |
|---------|---------|---------------|
| **Arithmetic mean** | $\bar{x} = \frac{1}{n}\sum x_i$ | Symmetric data, no extreme outliers |
| **Median** | Middle value when sorted | Skewed data, robust to outliers |
| **Mode** | Most frequent value | Categorical or multimodal data |
| **Geometric mean** | $\bar{x}_{geo} = \left(\prod x_i\right)^{1/n}$ | Compounding rates, multi-period returns |
| **Harmonic mean** | $\bar{x}_{harm} = \frac{n}{\sum 1/x_i}$ | Rates and ratios (P/E ratios, speed) |

In financial analysis, the choice of central tendency measure can dramatically change conclusions. For multi-year returns, using the arithmetic mean instead of the geometric mean systematically overstates compound performance — especially when volatility is high.

> **Key Concept:** **Geometric mean ≤ Arithmetic mean**, with equality only when all values are equal. The gap widens with volatility — this is why a "consistent 8% return" is mathematically more valuable than a "volatile 8% on average" return for long-term wealth accumulation.

In [ ]:
def arithmetic_mean(x):
    """Arithmetic mean from scratch."""
    x = np.asarray(x, dtype=float)
    return np.sum(x) / len(x)


def weighted_mean(x, w):
    """Weighted mean. Weights must sum to 1."""
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    assert np.isclose(np.sum(w), 1.0), "Weights must sum to 1"
    return np.sum(w * x)


def geometric_mean_return(returns):
    """Geometric mean of returns (not raw values).
    returns : array of periodic returns (e.g., 0.10 for 10%)
    """
    returns = np.asarray(returns, dtype=float)
    n = len(returns)
    return np.prod(1 + returns) ** (1 / n) - 1


def harmonic_mean(x):
    """Harmonic mean from scratch."""
    x = np.asarray(x, dtype=float)
    return len(x) / np.sum(1 / x)


def median(x):
    """Median from scratch."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n % 2 == 1:
        return x[n // 2]
    else:
        return (x[n // 2 - 1] + x[n // 2]) / 2


def mode(x):
    """Mode from scratch (returns most frequent value)."""
    x = np.asarray(x)
    values, counts = np.unique(x, return_counts=True)
    return values[np.argmax(counts)]


# ── Example: Annual returns of a fund
annual_returns = np.array([0.15, -0.08, 0.12, 0.06, -0.03, 0.18, 0.09, -0.01, 0.14, 0.07])

am = arithmetic_mean(annual_returns)
gm = geometric_mean_return(annual_returns)
med = median(annual_returns)

print("Annual returns:", [f"{r:.0%}" for r in annual_returns])
print(f"\nArithmetic mean: {am:.4%}")
print(f"Geometric mean:  {gm:.4%}")
print(f"Median:          {med:.4%}")
print(f"\nNote: Geometric mean < Arithmetic mean (always true for variable returns)")
print(f"Difference: {am - gm:.4%}")

# ── Portfolio weighted return
asset_returns = np.array([0.12, 0.08, 0.15])
weights = np.array([0.5, 0.3, 0.2])
port_ret = weighted_mean(asset_returns, weights)
print(f"\nPortfolio return (weighted): {port_ret:.4%}")

# ── Harmonic mean example: dollar-cost averaging
prices = np.array([10, 12, 8, 11, 9])
hm = harmonic_mean(prices)
am_p = arithmetic_mean(prices)
print(f"\nShare prices: {prices}")
print(f"Harmonic mean (avg cost basis): ${hm:.4f}")
print(f"Arithmetic mean:                ${am_p:.4f}")
print(f"H_mean <= A_mean: {hm <= am_p}")

**Interpreting the results:**

- The **arithmetic mean** (6.9%) is our best forecast for any single year's return.
- The **geometric mean** (6.55%) is the actual compound growth rate. If you invested $1,000, it grew at 6.55% per year on average.
- The **gap** between them (0.35%) is caused by volatility drag. Higher volatility would make this gap larger.
- The **harmonic mean** of prices ($9.80) is less than the arithmetic mean ($10.00), confirming that dollar-cost averaging naturally produces a better average price.

> **Common Mistake:** Reporting the arithmetic mean as the "average annual return" when evaluating a track record. A hedge fund that earned +100% in year 1 and -50% in year 2 has a 25% arithmetic mean return but a 0% geometric mean -- investors broke even! Always ask: "Is this the arithmetic or geometric mean?"**Interpreting the results:**

### Mean vs median — when each matters

The arithmetic mean is the most common measure of central tendency, but it has a serious weakness: **sensitivity to outliers**. A single extreme value can dramatically shift the mean.

**Example:** Five hedge fund returns are 5%, 6%, 7%, 8%, and 100%. The mean is 25.2%, but four of the five funds returned less than 10%. The mean is misleading.

The **median** (the middle value when sorted) is more robust:
* Five values, median = 7%
* Even with the 100% outlier, the median is unaffected

### When to use which

| Situation | Best Measure |
|-----------|--------------|
| Symmetric distribution (no outliers) | Mean |
| Skewed distribution (income, wealth, returns) | Median |
| Compound returns over time | Geometric mean |
| Comparing rates (yields, rates of change) | Harmonic mean |

### The geometric mean for returns

For multi-period returns, the **geometric mean** is mathematically correct:

$$\bar{r}_{geo} = \left[\prod_{i=1}^{n}(1 + r_i)\right]^{1/n} - 1$$

This represents the per-period compound rate that, applied $n$ times, produces the same final wealth as the actual sequence of returns.

> **CFA Exam Tip:** A common exam trap: question asks for the "average return" of a portfolio. If the period is multi-year and the returns are compounded, use the geometric mean. The arithmetic mean *overstates* compound returns — and the bias grows with volatility. For 50% gain followed by 50% loss, arithmetic mean is 0% but the actual outcome is -25% (geometric mean is -13.4%).

---
## 2. Measures of Dispersion: Quantifying Risk

### The Big Question: "How Spread Out Are Returns?"

Knowing the average return is not enough. Consider two funds:
- **Fund A:** Returns of 15%, -8%, 12%, 6%, -3% (average: 4.4%)
- **Fund B:** Returns of 5%, 4%, 5%, 4%, 4% (average: 4.4%)

Same average, but Fund A is much more volatile -- its returns swing wildly. Fund B is steady and predictable. As an investor, you'd probably prefer Fund B (assuming you're risk-averse, which most people are). This is why we need measures of **dispersion** (spread) to quantify risk.

### Variance and Standard Deviation

**Variance** measures the average squared deviation from the mean:

**Population variance** (when you have the entire population):
$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i - \mu)^2$$

**Sample variance** (when you have a sample -- the usual case):
$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

**Standard deviation** is the square root of variance: $\sigma = \sqrt{\sigma^2}$ or $s = \sqrt{s^2}$.

**Why $n-1$ (Bessel's correction)?** When estimating variance from a sample, dividing by $n$ would systematically underestimate the true variance. Dividing by $n-1$ corrects for this bias. The intuition: we used one "degree of freedom" to estimate the mean, so we have only $n-1$ independent pieces of information left.

**Investment meaning:** Standard deviation is the most common measure of **investment risk**. A stock with 30% annual standard deviation is riskier than one with 15%. In a normal distribution, about 68% of returns fall within one standard deviation of the mean.

**Worked Example:** For returns of 10%, 5%, 15%:
- Mean = (10 + 5 + 15)/3 = 10%
- Deviations: (0%, -5%, 5%)
- Squared deviations: (0, 25, 25)
- Sample variance = (0 + 25 + 25)/(3-1) = 25
- Standard deviation = 5%

### Mean Absolute Deviation (MAD)

$$\text{MAD} = \frac{1}{n}\sum_{i=1}^{n}|x_i - \bar{x}|$$

MAD is simpler and more robust to outliers than variance (since it doesn't square the deviations), but it's used less frequently in finance.

### Coefficient of Variation (CV): Comparing Apples to Oranges

$$CV = \frac{s}{\bar{x}}$$

**Investment meaning:** CV measures **risk per unit of return**. It allows comparison across investments with different expected returns. A fund with 10% return and 5% std dev (CV = 0.5) is "less risky per unit of return" than one with 20% return and 15% std dev (CV = 0.75).

### Sharpe Ratio: The Gold Standard of Risk-Adjusted Performance

$$\text{Sharpe} = \frac{\bar{R}_p - R_f}{\sigma_p}$$

where $R_f$ is the risk-free rate.

**Investment meaning:** The Sharpe ratio tells you how much **excess return** (above the risk-free rate) you earn per unit of risk. Higher is better. A Sharpe ratio of 1.0 means you earn 1% of excess return for each 1% of volatility -- that's considered good.

> **Key Concept:** Risk in finance is almost always measured by **standard deviation** (or variance). When someone says "this stock is risky," they mean its returns have high standard deviation -- they bounce around a lot.

> **CFA Exam Tip:** Be careful about whether a problem gives you population or sample data. Population uses $N$ in the denominator; sample uses $n-1$. On the exam, investment returns are almost always treated as **sample** data.

Let's compare two funds to see these measures in action.

### Why variance and standard deviation matter so much in finance

Variance (and its square root, standard deviation) is the foundation of risk measurement in modern finance:

* **Portfolio theory:** Markowitz mean-variance optimisation uses variance as the risk measure
* **Sharpe ratio:** Risk-adjusted return uses standard deviation in the denominator
* **Value at Risk (VaR):** Often computed as $\mu - 1.645\sigma$ (95% confidence) or $\mu - 2.326\sigma$ (99%)
* **Black-Scholes:** Volatility ($\sigma$) is one of the five inputs to option pricing
* **Risk parity portfolios:** Allocate so each asset contributes equal variance

### Variance is *not* the only measure of risk

Despite its prominence, variance has limitations:

1. **Treats upside and downside symmetrically.** A portfolio that has high volatility because of *positive* surprises is treated the same as one with high volatility from losses. Investors care more about downside.

2. **Assumes returns are normal.** When returns have fat tails, variance underestimates extreme losses.

3. **Two-moment summary.** Variance ignores skewness and kurtosis — important features of real return distributions.

Alternative risk measures (downside deviation, semi-variance, VaR, expected shortfall) address these limitations but are more complex to compute and interpret.

> **CFA Exam Tip:** The CFA curriculum tests both standard deviation as the conventional risk measure AND the alternatives. Be prepared to compute downside deviation, semi-variance, and the Sortino ratio (uses downside deviation instead of total volatility). The choice of risk measure can significantly change rankings when comparing investment alternatives.

In [ ]:
def sample_variance(x):
    """Sample variance with Bessel's correction."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = np.sum(x) / n
    return np.sum((x - mean) ** 2) / (n - 1)


def sample_std(x):
    """Sample standard deviation."""
    return np.sqrt(sample_variance(x))


def population_variance(x):
    """Population variance (no Bessel's correction)."""
    x = np.asarray(x, dtype=float)
    mean = np.sum(x) / len(x)
    return np.sum((x - mean) ** 2) / len(x)


def mean_abs_deviation(x):
    """Mean absolute deviation."""
    x = np.asarray(x, dtype=float)
    mean = np.sum(x) / len(x)
    return np.sum(np.abs(x - mean)) / len(x)


def coeff_of_variation(x):
    """Coefficient of variation = std / mean."""
    return sample_std(x) / arithmetic_mean(x)


def sharpe_ratio(returns, rf=0.02):
    """Sharpe ratio = (mean return - risk-free rate) / std."""
    return (arithmetic_mean(returns) - rf) / sample_std(returns)


# ── Example with two funds
fund_A = np.array([0.15, -0.08, 0.12, 0.06, -0.03, 0.18, 0.09, -0.01, 0.14, 0.07])
fund_B = np.array([0.05, 0.03, 0.07, 0.02, 0.04, 0.06, 0.03, 0.05, 0.04, 0.06])

print(f"{'Measure':<25} {'Fund A':>10} {'Fund B':>10}")
print("-" * 47)
print(f"{'Mean return':<25} {arithmetic_mean(fund_A):>10.4%} {arithmetic_mean(fund_B):>10.4%}")
print(f"{'Sample Std Dev':<25} {sample_std(fund_A):>10.4%} {sample_std(fund_B):>10.4%}")
print(f"{'MAD':<25} {mean_abs_deviation(fund_A):>10.4%} {mean_abs_deviation(fund_B):>10.4%}")
print(f"{'Coeff of Variation':<25} {coeff_of_variation(fund_A):>10.4f} {coeff_of_variation(fund_B):>10.4f}")
print(f"{'Sharpe (rf=2%)':<25} {sharpe_ratio(fund_A):>10.4f} {sharpe_ratio(fund_B):>10.4f}")

# Verify against numpy
assert np.isclose(sample_variance(fund_A), np.var(fund_A, ddof=1)), "Variance mismatch!"
print("\n✓ All implementations verified against numpy")

**Interpreting the comparison:**

Fund A has a higher mean return (6.9% vs 4.5%), but also much higher risk (standard deviation of 8.8% vs 1.5%). The question is: does the extra return justify the extra risk?

- **Coefficient of Variation:** Fund A's CV is 1.27 vs Fund B's 0.33. This means Fund A has almost 4 times the risk per unit of return. By this measure, Fund B is more efficient.
- **Sharpe Ratio:** Fund A's Sharpe is 0.56 vs Fund B's 1.67. Fund B delivers 3 times more excess return per unit of risk! Despite Fund A's higher raw return, **Fund B is the better risk-adjusted performer**.

This illustrates a crucial lesson: raw returns can be misleading. A fund that earned 20% by taking enormous risks might be worse than one that earned 8% with very low risk. Risk-adjusted metrics like the Sharpe ratio tell the true story.

> **Common Mistake:** Comparing funds only by their average returns without considering risk. A fund with 15% return and 40% standard deviation is probably worse than one with 8% return and 10% standard deviation.**Interpreting the comparison:**

### Sample variance vs population variance — the $n-1$ correction

Two formulas exist for variance:

**Population variance** (when you have all data):
$$\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \mu)^2$$

**Sample variance** (when you have a sample):
$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

The denominator differs: $n$ for population, $n-1$ for sample. This $n-1$ adjustment is **Bessel's correction**, and it makes the sample variance an *unbiased estimator* of the population variance.

### Why the correction matters

When computing sample variance, you use the *sample mean* $\bar{x}$ rather than the true population mean $\mu$. Because $\bar{x}$ is itself derived from the sample, the sum of squared deviations from $\bar{x}$ is systematically *smaller* than the sum from $\mu$. Dividing by $n-1$ instead of $n$ corrects for this bias.

> **Common Mistake:** When working with financial returns, always use the sample formula ($n-1$ denominator). Statistical software defaults vary: NumPy's `np.var()` defaults to population (n), while `np.var(x, ddof=1)` gives the sample variance. Always check which version your tool uses.

### Standard deviation — the practical statistic

The standard deviation $\sigma$ (or $s$) is the square root of variance. It has the same units as the underlying variable, making it more intuitive than variance:

* Variance of returns: %² (hard to interpret)
* Standard deviation of returns: % (directly comparable to mean return)

This is why finance practitioners almost always cite standard deviation rather than variance.

---
## 3. Skewness & Kurtosis: Beyond Mean and Variance

### Why Mean and Variance Aren't Enough

Mean and standard deviation describe the "center" and "width" of a distribution, but they miss two critical features that matter enormously for investors: the **asymmetry** and **tail thickness** of returns.

Consider this: two distributions can have the same mean and standard deviation but very different shapes. One might have a long left tail (frequent small gains, rare catastrophic losses), while the other is perfectly symmetric. An investor would strongly prefer to know this!

### Skewness (Third Standardized Moment): Asymmetry

$$S = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^3$$

Skewness measures **asymmetry** in the distribution:

- **$S > 0$ (Right/positive skew):** The right tail is longer. This means occasional large gains but more frequent small losses. *Lottery tickets* have positive skew.
- **$S < 0$ (Left/negative skew):** The left tail is longer. This means frequent small gains but occasional large losses. **Equity returns tend to be negatively skewed** -- markets crash more than they boom.
- **$S = 0$:** The distribution is symmetric (like the normal distribution).

**Why investors care:** Negative skewness means the downside risk is worse than you'd expect from looking at just the standard deviation. A portfolio with -1.0 skewness has "hidden" crash risk that standard deviation doesn't capture.

### Kurtosis (Fourth Standardized Moment): Tail Thickness

$$K = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^4$$

**Excess kurtosis** = $K - 3$ (we subtract 3 because the normal distribution has kurtosis of 3).

- **Excess kurtosis > 0 (Leptokurtic):** **Fat tails** -- extreme events occur more often than a normal distribution predicts. Most financial return distributions are leptokurtic.
- **Excess kurtosis = 0 (Mesokurtic):** Normal-like tails.
- **Excess kurtosis < 0 (Platykurtic):** Thin tails -- extreme events are rarer than normal.

**Why investors care:** Fat tails mean that "once in a century" events (like the 2008 financial crisis) happen much more often than once a century. Standard risk models based on the normal distribution systematically *underestimate* the probability of extreme losses.

> **Key Concept:** Real financial returns typically exhibit **negative skewness** (crash risk) and **positive excess kurtosis** (fat tails). This combination is the worst possible for investors: large losses are both more likely (fat tails) and more extreme (negative skew) than normal models predict.

> **CFA Exam Tip:** Remember the mnemonic: "Leptokurtic = Leaping tails" (fat tails that seem to leap beyond what's normal). Excess kurtosis > 0 means more extreme observations than the normal distribution.

Let's generate three different return distributions and compare their shapes.

### The four moments — a compact distribution summary

A surprising mathematical fact: the first four moments (mean, variance, skewness, kurtosis) capture most of what matters about a distribution shape. While they don't fully determine the distribution, they convey enough for risk management decisions.

**Why four and not more?** In principle, every distribution has infinitely many moments (5th, 6th, etc.). But:
* Higher moments are increasingly difficult to estimate accurately (need huge samples)
* They contribute progressively less new information about the distribution shape
* They are harder to interpret economically

In practice, focusing on the first four moments balances information vs reliability. This is sometimes called the **four-moment expansion** or **Cornish-Fisher expansion** in risk management.

> **Common Mistake:** Beginners sometimes report "mean and standard deviation" as if these fully describe a distribution. They do — *only* if the distribution is normal. For real financial data, including skewness and kurtosis is essential for accurate risk assessment.

In [ ]:
def skewness(x):
    """Sample skewness (third standardized moment)."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = np.sum(x) / n
    std = np.sqrt(np.sum((x - mean) ** 2) / (n - 1))
    return np.sum(((x - mean) / std) ** 3) * n / ((n - 1) * (n - 2))


def kurtosis(x, excess=True):
    """Sample kurtosis. If excess=True, subtract 3 (normal = 0)."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = np.sum(x) / n
    std = np.sqrt(np.sum((x - mean) ** 2) / (n - 1))
    m4 = np.sum(((x - mean) / std) ** 4) / n
    # Adjusted formula for sample excess kurtosis
    raw = (n * (n + 1)) / ((n - 1) * (n - 2) * (n - 3)) * np.sum(((x - mean) / std) ** 4)
    adjustment = 3 * (n - 1) ** 2 / ((n - 2) * (n - 3))
    if excess:
        return raw - adjustment
    return raw - adjustment + 3


# ── Generate financial return distributions
n_samples = 10_000
normal_returns = rng.normal(0.08, 0.18, n_samples)
# Left-skewed: simulate crash-prone returns
left_skewed = -rng.lognormal(-0.08, 0.3, n_samples) + 0.20
# Fat-tailed: t-distribution with few degrees of freedom
fat_tailed = rng.standard_t(df=4, size=n_samples) * 0.18 / np.sqrt(4 / (4 - 2)) + 0.08

distributions = {
    'Normal returns': normal_returns,
    'Left-skewed': left_skewed,
    'Fat-tailed (t, df=4)': fat_tailed,
}

print(f"{'Distribution':<25} {'Mean':>8} {'Std':>8} {'Skew':>8} {'Ex Kurt':>8}")
print("-" * 60)
for name, data in distributions.items():
    print(f"{name:<25} {arithmetic_mean(data):>8.4f} {sample_std(data):>8.4f} "
          f"{skewness(data):>8.4f} {kurtosis(data):>8.4f}")

**Interpreting the moments:**

- **Normal returns:** Skewness near 0, excess kurtosis near 0 -- this is our baseline.
- **Left-skewed:** Negative skewness means the distribution has a long left tail. This models equity returns well -- markets tend to crash more violently than they rally.
- **Fat-tailed:** High excess kurtosis means extreme events (both up and down) happen more often than normal. The t-distribution with 4 degrees of freedom is a classic model for this.

The histograms below make these differences visually obvious.**Interpreting the moments:**

### Higher moments — beyond mean and variance

The first four central moments characterise different aspects of a distribution:

| Moment | Name | What it Measures |
|--------|------|------------------|
| 1st | Mean | Central tendency |
| 2nd | Variance | Spread / dispersion |
| 3rd | Skewness | Asymmetry |
| 4th | Kurtosis | Tail heaviness / peakedness |

### Skewness — asymmetry

Standardised skewness:

$$S = \frac{1}{n}\sum_{i=1}^{n}\left(\frac{x_i - \bar{x}}{s}\right)^3$$

* **$S > 0$** (positive skew): Long right tail. Mean > Median. Examples: lottery winnings, hedge fund returns.
* **$S = 0$** (symmetric): No skew. Examples: normal distribution.
* **$S < 0$** (negative skew): Long left tail. Mean < Median. Examples: stock returns during crashes, bond defaults.

### Kurtosis — tail heaviness

Standardised kurtosis:

$$K = \frac{1}{n}\sum_{i=1}^{n}\left(\frac{x_i - \bar{x}}{s}\right)^4$$

* Normal distribution has $K = 3$
* **Excess kurtosis** $= K - 3$ measures deviation from normal
* **$K > 3$** (leptokurtic / "fat tails"): More extreme outcomes than normal. Examples: financial returns, especially crisis periods.
* **$K < 3$** (platykurtic / "thin tails"): Fewer extreme outcomes. Examples: uniform distribution.

### Why this matters in finance

Stock returns are well-known to exhibit:
* **Negative skewness:** Crashes are larger than rallies (in absolute terms)
* **High kurtosis:** Tails are fatter than the normal distribution predicts

This means standard tools (which often assume normality) **systematically underestimate** the probability of extreme losses. This insight motivated the development of *Value at Risk*, *Expected Shortfall*, and *fat-tail-aware* risk measures.

> **CFA Exam Tip:** A common CFA exam scenario: a portfolio with normally-distributed returns vs one with negatively skewed and high kurtosis. Even with identical mean and variance, the second portfolio has substantially higher tail risk. The exam tests whether you recognise that mean-variance analysis alone is insufficient when distributions deviate from normality.

In [ ]:
# ── Visualization: Distribution shapes
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [PRIMARY, SECONDARY, TERTIARY]

for ax, (name, data), color in zip(axes, distributions.items(), colors):
    ax.hist(data, bins=80, density=True, alpha=0.7, color=color, edgecolor='white')
    ax.set_title(f'{name}\nSkew={skewness(data):.2f}, ExKurt={kurtosis(data):.2f}')
    ax.set_xlabel('Return')
    ax.set_ylabel('Density')
    ax.axvline(arithmetic_mean(data), color='black', linestyle='--', linewidth=1.5, label='Mean')
    ax.legend()

plt.suptitle('Skewness and Kurtosis in Return Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Reading the histograms:**

- **Normal (left):** Symmetric bell shape. The mean sits right at the center. Tails are thin.
- **Left-skewed (center):** The bulk of returns are on the right (positive returns), but there's a long tail stretching to the left (large losses). The mean is pulled to the left of the peak. This is what equity market returns look like in practice.
- **Fat-tailed (right):** Looks symmetric like the normal, but notice the tails extend further. Extreme returns (both positive and negative) are more common. This is why "Black Swan" events happen more often than normal models predict.

> **Common Mistake:** Assuming returns are normally distributed when making risk calculations. Real returns have fatter tails and negative skewness. This means VaR (Value at Risk) calculated using the normal distribution will **underestimate** the true risk of extreme losses.**Reading the histograms:**

### Distinguishing distribution shapes by eye

The histogram visualisations reveal characteristic shapes:

1. **Symmetric, bell-shaped:** Normal distribution. Peak in the middle, equal tails on both sides.

2. **Right-skewed:** Long right tail with most density on the left. Common in income, asset prices, lognormal-distributed variables.

3. **Left-skewed:** Long left tail with most density on the right. Common in equity returns during crashes.

4. **Heavy-tailed:** More density in the extremes than a normal distribution would predict. Visible as "fatter" tails on the histogram.

### Quantitative diagnostics

Rather than rely on visual inspection alone, statisticians use:

* **Skewness > 1 or < -1:** Highly skewed
* **Excess kurtosis > 3:** Heavy-tailed (sometimes called "fat-tailed")
* **Jarque-Bera test:** Combines skewness and kurtosis into a single normality test
* **QQ-plot (quantile-quantile):** Compares sample quantiles to theoretical quantiles — deviations from a straight line indicate non-normality

> **Key Concept:** Returns of major equity indices are *not* normally distributed — they have negative skewness and high kurtosis (leptokurtic). This is one of the most well-documented empirical regularities in finance. Models that assume normality (such as the original Black-Scholes) systematically underprice tail risk.

---
## 4. Probability Distributions

A probability distribution describes all possible outcomes and their likelihoods. Different situations call for different distributions. Here are the key ones for finance.

### Discrete Distributions (Countable Outcomes)

**Binomial Distribution:** Models the number of "successes" in $n$ independent trials, each with probability $p$.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Finance application:** "Out of 20 stocks in a portfolio, what's the probability that exactly 12 outperform the benchmark?" If each stock has a 40% chance of outperforming independently, this is binomial.

**Poisson Distribution:** Models the number of rare events in a fixed interval.

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

**Finance application:** "How many corporate defaults will occur in a credit portfolio this year?" If defaults are rare and approximately independent, the count follows a Poisson distribution.

### Continuous Distributions

**Normal (Gaussian) Distribution:** The most important distribution in statistics.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**Finance application:** Short-term stock returns are approximately normal. Many risk models assume normality.

**Lognormal Distribution:** If $\ln(X) \sim N(\mu, \sigma^2)$, then $X$ is lognormal.

**Finance application:** Stock *prices* (not returns) follow a lognormal distribution. This makes sense because prices can't be negative, and lognormal distributions are bounded below by zero.

**Student's t-Distribution:** Like the normal but with fatter tails. Used when sample sizes are small and population variance is unknown.

**Finance application:** Testing whether a fund manager has significant alpha with limited data (e.g., 30 months of returns).

| Distribution | Normal vs t | When to use |
|---|---|---|
| Normal (z) | Thinner tails | Large samples OR known population variance |
| Student's t | Fatter tails | Small samples AND unknown population variance |

As the sample size (degrees of freedom) increases, the t-distribution converges to the normal. With 30+ observations, they're nearly identical.

> **CFA Exam Tip:** When in doubt about which distribution to use for hypothesis testing: if you know the population standard deviation, use z. If you're estimating it from the sample, use t. In practice, you almost always use t.

Let's implement these distributions and visualize them.

### Discrete vs continuous distributions

Distributions are fundamentally divided into two types:

**Discrete distributions** describe variables that can take a countable number of values:
* Binomial (number of successes in $n$ trials)
* Poisson (number of events in a fixed interval)
* Geometric (trials until first success)
* Negative binomial (trials until $r$-th success)

**Continuous distributions** describe variables that can take any value in a range:
* Normal (most common — symmetric, bell-shaped)
* Lognormal (right-skewed, positive values only)
* Exponential (memoryless, time-to-event)
* Uniform (equal probability across a range)
* t-distribution (heavier tails than normal — used for small samples)
* Chi-squared, F (used for hypothesis testing)

### The role of the Central Limit Theorem

The CLT is one of the most important results in statistics:

> **Theorem (CLT):** As sample size $n$ grows, the distribution of the sample mean approaches a normal distribution, regardless of the underlying distribution shape (subject to mild conditions).

Implications for finance:
1. Even if individual stock returns are non-normal, the *average* return across many stocks is approximately normal
2. Even if individual periods have skewed returns, *long-run* compound returns are approximately lognormal
3. Statistical inference using normal-based methods (z-tests, t-tests) is robust *for averages*, even when raw data is non-normal

> **CFA Exam Tip:** The CLT is heavily tested. Remember its key claim: averages are normally distributed even when individual observations are not. The standard error of the mean is $\sigma/\sqrt{n}$ — meaning you need 4× the sample size to halve the standard error. This is why large samples are so valuable in statistical estimation.

In [ ]:
def binomial_pmf(k, n, p):
    """Binomial PMF from scratch.
    P(X=k) = C(n,k) * p^k * (1-p)^(n-k)
    """
    from math import lgamma
    log_coeff = lgamma(n + 1) - lgamma(k + 1) - lgamma(n - k + 1)
    log_pmf = log_coeff + k * np.log(p) + (n - k) * np.log(1 - p)
    return np.exp(log_pmf)


def poisson_pmf(k, lam):
    """Poisson PMF from scratch.
    P(X=k) = lambda^k * e^(-lambda) / k!
    """
    from math import lgamma
    log_pmf = k * np.log(lam) - lam - lgamma(k + 1)
    return np.exp(log_pmf)


def normal_pdf(x, mu=0, sigma=1):
    """Normal PDF from scratch."""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def normal_cdf(x, mu=0, sigma=1):
    """Normal CDF using error function.
    Phi(x) = 0.5 * (1 + erf((x - mu) / (sigma * sqrt(2))))
    """
    from scipy.special import erf
    return 0.5 * (1 + erf((x - mu) / (sigma * np.sqrt(2))))


# ── Verify against scipy
x_test = 1.96
print(f"Normal PDF at x={x_test}: ours={normal_pdf(x_test):.6f}, scipy={stats.norm.pdf(x_test):.6f}")
print(f"Normal CDF at x={x_test}: ours={normal_cdf(x_test):.6f}, scipy={stats.norm.cdf(x_test):.6f}")
print(f"Binomial P(X=3|n=10,p=0.3): ours={binomial_pmf(3, 10, 0.3):.6f}, scipy={stats.binom.pmf(3, 10, 0.3):.6f}")
print(f"Poisson P(X=5|lambda=3): ours={poisson_pmf(5, 3):.6f}, scipy={stats.poisson.pmf(5, 3):.6f}")

Our from-scratch implementations match scipy perfectly. Now let's visualize these distributions to build intuition for their shapes.Our from-scratch implementations match scipy perfectly. Now let's visualize these distributions.

### The Big Three — distributions every analyst must know

**1. Normal distribution** $N(\mu, \sigma^2)$
* Continuous, symmetric, defined on the real line
* Two parameters: mean and variance
* Sums and averages of many independent random variables converge to normal (Central Limit Theorem)
* Foundation of most parametric statistical methods

**2. Lognormal distribution** $\text{LN}(\mu, \sigma^2)$
* Variable $X$ is lognormal iff $\log X$ is normal
* Right-skewed, defined on positive real numbers
* Used to model stock prices, asset values, and other strictly positive quantities

**3. Binomial distribution** $\text{Bin}(n, p)$
* Discrete, defined on $\{0, 1, ..., n\}$
* Number of "successes" in $n$ independent Bernoulli trials with success probability $p$
* Foundation for binomial option pricing, default modelling, and many credit risk applications

### Choosing the right distribution

| Scenario | Distribution |
|----------|--------------|
| Stock prices | Lognormal |
| Stock returns (log returns) | Normal (approximately) |
| Bond default counts | Binomial or Poisson |
| Time between events | Exponential |
| Correlated returns | Multivariate normal (or with copula) |
| Extreme losses | Generalised extreme value (GEV) |

> **CFA Exam Tip:** Distribution selection is rarely tested directly, but understanding which distribution applies to which financial variable is essential. A common trap: questions about returns might assume normality, but questions about prices should use lognormality.

In [ ]:
# ── Visualization: Distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Binomial
n_binom, p_binom = 20, 0.4
k_vals = np.arange(0, n_binom + 1)
binom_probs = [binomial_pmf(k, n_binom, p_binom) for k in k_vals]
axes[0, 0].bar(k_vals, binom_probs, color=PRIMARY, alpha=0.8, edgecolor='white')
axes[0, 0].set_title(f'Binomial(n={n_binom}, p={p_binom})')
axes[0, 0].set_xlabel('k'); axes[0, 0].set_ylabel('P(X=k)')

# Poisson
lam = 5
k_pois = np.arange(0, 20)
pois_probs = [poisson_pmf(k, lam) for k in k_pois]
axes[0, 1].bar(k_pois, pois_probs, color=SECONDARY, alpha=0.8, edgecolor='white')
axes[0, 1].set_title(f'Poisson(lambda={lam})')
axes[0, 1].set_xlabel('k'); axes[0, 1].set_ylabel('P(X=k)')

# Normal vs t-distribution
x_range = np.linspace(-4, 4, 300)
axes[1, 0].plot(x_range, normal_pdf(x_range), color=PRIMARY, linewidth=2, label='N(0,1)')
for df in [3, 5, 30]:
    axes[1, 0].plot(x_range, stats.t.pdf(x_range, df), linewidth=1.5, 
                    linestyle='--', label=f't(df={df})')
axes[1, 0].set_title('Normal vs t-Distribution')
axes[1, 0].legend(); axes[1, 0].set_xlabel('x'); axes[1, 0].set_ylabel('f(x)')

# Lognormal
x_ln = np.linspace(0.01, 5, 300)
for sigma in [0.25, 0.5, 1.0]:
    pdf_ln = (1 / (x_ln * sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * (np.log(x_ln) / sigma) ** 2)
    axes[1, 1].plot(x_ln, pdf_ln, linewidth=2, label=f'sigma={sigma}')
axes[1, 1].set_title('Lognormal Distribution')
axes[1, 1].legend(); axes[1, 1].set_xlabel('x'); axes[1, 1].set_ylabel('f(x)')
axes[1, 1].set_ylim(0, 2)

plt.tight_layout()
plt.show()

**Reading the distribution plots:**

- **Binomial (top-left):** With n=20 and p=0.4, the distribution centers around 8 (which is $np = 20 \times 0.4$). It's already starting to look bell-shaped -- the binomial approaches the normal for large n.

- **Poisson (top-right):** The distribution is right-skewed, peaking near the mean of 5. Good for modeling rare events like defaults.

- **Normal vs t (bottom-left):** This is a critical comparison. The t-distribution with few degrees of freedom (df=3) has noticeably fatter tails -- more probability in the extremes. By df=30, the t-distribution is nearly identical to the normal. This is why the choice between z-test and t-test matters mainly for small samples.

- **Lognormal (bottom-right):** Bounded below by zero and right-skewed. Higher sigma makes it more skewed. Stock prices follow this distribution because they can't go negative.**Reading the distribution plots:**

### Shape characteristics by distribution

* **Normal:** Smooth, symmetric, bell-shaped. Tails decay rapidly (exponentially squared).

* **Lognormal:** Right-skewed. Bunched near zero with a long right tail. Cannot be negative.

* **Binomial:** Discrete, with bars at integer values. Shape depends on $p$:
  - $p = 0.5$: symmetric
  - $p < 0.5$: right-skewed
  - $p > 0.5$: left-skewed
  - As $n$ grows, becomes increasingly normal-like (CLT in action)

### Connection between distributions

Many seeming relationships exist:
* If $X \sim N(\mu, \sigma^2)$, then $e^X \sim \text{LN}(\mu, \sigma^2)$
* If $X_i \sim \text{Bernoulli}(p)$, then $\sum X_i \sim \text{Bin}(n, p)$
* As $n \to \infty$ with $p$ fixed, $\text{Bin}(n, p) \to N(np, np(1-p))$
* Sum of $k$ independent normals is normal with summed parameters

These connections allow analysts to use the most tractable representation for each problem.

> **Common Mistake:** Students sometimes forget the lognormal's *right-skewness*. A lognormal distribution with median \$100 has a *mean* greater than \$100 — the right tail pulls the average up. This is why average wealth in a population is typically much higher than median wealth.

---
## 5. Normal Distribution Deep Dive

The normal distribution is the workhorse of quantitative finance. You need to know it inside and out.

### Standard Normal & Z-Scores

Any normal variable can be **standardized** by subtracting the mean and dividing by the standard deviation:

$$Z = \frac{X - \mu}{\sigma} \sim N(0, 1)$$

**Investment meaning:** A z-score tells you how many standard deviations a return is from the mean. A z-score of -2 means the return was 2 standard deviations below average -- a very bad outcome.

### The Empirical Rule (68-95-99.7)

For any normal distribution:
- **68.27%** of observations fall within $\pm 1\sigma$ of the mean
- **95.45%** fall within $\pm 2\sigma$
- **99.73%** fall within $\pm 3\sigma$

**Worked Example:** A stock has expected return of 10% and standard deviation of 20%.
- There's a 68% chance the return falls between -10% and 30% (one sigma)
- There's a 95% chance the return falls between -30% and 50% (two sigma)
- A return below -50% (three sigma below the mean) has only a 0.13% chance

### Roy's Safety-First Criterion

This practical tool helps investors choose the portfolio with the **lowest probability of falling below a minimum acceptable return** $R_L$ (the "safety level").

$$\text{SFRatio} = \frac{E(R_p) - R_L}{\sigma_p}$$

**Decision rule:** Choose the portfolio with the **highest** SFRatio (it has the smallest probability of disaster).

**Intuition:** The SFRatio measures how many standard deviations the expected return is above the danger zone. More standard deviations = safer.

> **CFA Exam Tip:** Roy's Safety-First ratio looks just like the Sharpe ratio, but with $R_L$ (threshold return) instead of $R_f$ (risk-free rate). If $R_L = R_f$, they give the same ranking.

Let's verify the empirical rule and work through a safety-first example.

### The empirical rule (68-95-99.7 rule)

For a normal distribution:
* **68%** of values fall within 1 standard deviation of the mean
* **95%** within 2 standard deviations
* **99.7%** within 3 standard deviations

This rule provides quick mental arithmetic for risk assessment. If a stock's daily returns have $\mu = 0\%, \sigma = 1\%$:
* 68% of days: return is between -1% and +1%
* 95% of days: return is between -2% and +2%
* Days with returns below -3% should occur only 0.15% of the time (roughly once every 3 years)

But financial data has *fat tails* — extreme moves occur far more often than the empirical rule predicts. The 1987 stock market crash (-22.6% in one day) was a 22-standard-deviation event under normality assumptions — virtually impossible. That such "impossible" events occur regularly in finance is a powerful warning against blind reliance on normality.

### Standardising: the Z-score

The Z-score converts any observation to its position in standard normal terms:

$$Z = \frac{X - \mu}{\sigma}$$

This standardisation enables:
* Comparing observations from different distributions (e.g., test scores from different exams)
* Computing probabilities using a single standard normal table
* Identifying outliers ($|Z| > 3$ is unusual under normality)

> **Common Mistake:** The empirical rule applies *only* to normal distributions. Applying it to non-normal data (especially financial returns) systematically underestimates tail risk. Always check the distribution shape before using normality-based intuitions.

In [ ]:
# ── Verify empirical rule
print("Empirical Rule Verification (Standard Normal):")
for k in [1, 2, 3]:
    prob = normal_cdf(k) - normal_cdf(-k)
    print(f"  P(-{k}sigma <= Z <= +{k}sigma) = {prob:.4%}")

# ── Z-score example
mu_ret, sigma_ret = 0.10, 0.20
threshold = 0.0  # break-even
z = (threshold - mu_ret) / sigma_ret
prob_loss = normal_cdf(z)
print(f"\nPortfolio: mu = {mu_ret:.0%}, sigma = {sigma_ret:.0%}")
print(f"Z-score for R < {threshold:.0%}: z = {z:.2f}")
print(f"P(loss) = P(R < 0) = {prob_loss:.4%}")

# ── Roy's Safety-First Criterion
R_L = 0.02  # minimum acceptable return

portfolios = [
    {'name': 'Portfolio A', 'mu': 0.12, 'sigma': 0.20},
    {'name': 'Portfolio B', 'mu': 0.10, 'sigma': 0.14},
    {'name': 'Portfolio C', 'mu': 0.08, 'sigma': 0.10},
]

print(f"\nRoy's Safety-First Criterion (R_L = {R_L:.0%}):\n")
print(f"{'Portfolio':<15} {'E(R)':>8} {'sigma':>8} {'SFRatio':>10} {'P(R<R_L)':>10}")
print("-" * 55)
for p in portfolios:
    sf = (p['mu'] - R_L) / p['sigma']
    prob = normal_cdf(-sf)  # P(R < R_L)
    print(f"{p['name']:<15} {p['mu']:>8.2%} {p['sigma']:>8.2%} {sf:>10.4f} {prob:>10.4%}")
    p['sf'] = sf

best = max(portfolios, key=lambda p: p['sf'])
print(f"\nBest by Safety-First: {best['name']} (highest SFRatio = {best['sf']:.4f})")

**Interpreting the Safety-First analysis:**

- **Portfolio A** has the highest expected return (12%), but also the highest risk (20% std dev). Its SFRatio is 0.50, meaning the expected return is only 0.5 standard deviations above the danger zone.
- **Portfolio C** has the lowest return (8%), but by far the lowest risk (10%). Its SFRatio is 0.60 -- the highest, making it the safest choice.

Portfolio C wins because its narrow return distribution keeps losses rare. Despite lower expected returns, it has only a 27.4% chance of falling below the 2% threshold, compared to 30.9% for Portfolio A.

The Z-score calculation also reveals something important: with a 10% expected return and 20% volatility, there's a **30.85% chance of losing money** in any given year. Risk is very real!**Interpreting the Safety-First analysis:**

### Roy's Safety-First criterion

Andrew Roy (1952) proposed an early "safety-first" approach to portfolio choice:

$$\text{Maximize } \quad \frac{E[R_P] - R_L}{\sigma_P}$$

where $R_L$ is a *minimum threshold return* (the "disaster level"). The portfolio that maximises this ratio minimises the probability of falling below the threshold (under normality).

The safety-first ratio is structurally identical to the **Sharpe ratio**:

$$\text{Sharpe} = \frac{E[R_P] - R_F}{\sigma_P}$$

The only difference: Sharpe uses the risk-free rate $R_F$ as the threshold; safety-first uses an arbitrary disaster level.

### When safety-first is preferred

For some investors (pension funds, insurance companies), the goal is not maximum return but *adequate funding*. Safety-first frames the problem correctly: how do I minimise the chance of failing to meet my obligations?

This is conceptually equivalent to **liability-driven investing** — match assets to liabilities to ensure obligations are always met.

> **CFA Exam Tip:** Roy's safety-first ratio appears in CFA Level 1 (descriptive statistics) and Level 3 (institutional portfolio management). The connection to VaR-style risk measures and downside protection is a recurring theme.

---
## 6. Hypothesis Testing

### The Big Idea

Hypothesis testing is a formal framework for making decisions based on data. In finance, we constantly ask questions like:
- "Does this fund manager actually have skill, or were the returns just luck?"
- "Has the average return of this asset changed since 2020?"
- "Is this trading strategy's alpha significantly different from zero?"

Hypothesis testing gives us a rigorous way to answer these questions while controlling for the possibility that we're fooled by random chance.

### The Step-by-Step Framework

1. **State the hypotheses:**
   - $H_0$ (null hypothesis): The "nothing special" claim (e.g., "the manager has no skill", $\mu = 0$)
   - $H_a$ (alternative hypothesis): What you're trying to show (e.g., "the manager has positive alpha", $\mu > 0$)

2. **Choose significance level $\alpha$:** Usually 0.05 (5%). This is the probability of rejecting $H_0$ when it's actually true (false positive rate).

3. **Compute the test statistic:** A number that measures how far the sample result is from what $H_0$ predicts.

4. **Determine the p-value:** The probability of getting a result this extreme (or more) if $H_0$ is true.

5. **Decision:** Reject $H_0$ if p-value < $\alpha$.

### Z-Test vs T-Test

| Test | Formula | When to Use |
|---|---|---|
| Z-test | $z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$ | Known population $\sigma$, or large $n$ |
| T-test | $t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$ | Unknown population $\sigma$ (use sample $s$) |

The key difference: $s$ has more uncertainty than $\sigma$, so the t-distribution has fatter tails, making it harder to reject $H_0$. This is appropriate -- with less information, we should be more cautious.

### Type I and Type II Errors

| | $H_0$ is true | $H_0$ is false |
|---|---|---|
| **Reject $H_0$** | Type I error ($\alpha$) | Correct! (Power = $1-\beta$) |
| **Fail to reject $H_0$** | Correct! | Type II error ($\beta$) |

- **Type I error ($\alpha$):** Concluding a manager has skill when they don't (false positive). You invest in a mediocre fund.
- **Type II error ($\beta$):** Failing to identify a skilled manager (false negative). You miss a great investment.
- **Power ($1-\beta$):** The probability of correctly detecting skill when it exists.

> **Key Concept:** In finance, Type I errors (investing based on false signals) are often costlier than Type II errors (missing opportunities). This is why we typically use a low significance level like 5% -- we want to be quite confident before acting on a signal.

> **Common Mistake:** Saying "we accept $H_0$" when the test is not significant. The correct phrasing is "we **fail to reject** $H_0$." Absence of evidence is not evidence of absence -- the manager might have skill, but our sample was too small to detect it.

Let's work through two complete examples.

### The logic of hypothesis testing — a courtroom analogy

Hypothesis testing operates like a criminal trial:

* **Null hypothesis $H_0$:** "The defendant is innocent" (the default assumption)
* **Alternative hypothesis $H_a$:** "The defendant is guilty"
* **Test statistic:** The evidence presented
* **P-value:** Probability of seeing this evidence (or more extreme) if the defendant is innocent
* **Significance level $\alpha$:** The threshold for "beyond reasonable doubt"

Just as we need strong evidence to convict (rejecting innocence), we need strong statistical evidence to reject $H_0$. Failing to reject $H_0$ doesn't mean the defendant is innocent — only that the evidence is insufficient to convict.

> **Key Concept:** The asymmetry between Type I and Type II errors mirrors the asymmetry in criminal law: we'd rather let some guilty go free (Type II error) than convict the innocent (Type I error). This is why the default $\alpha = 0.05$ controls Type I errors strictly while accepting some Type II errors.

In [ ]:
def z_test(sample_mean, mu_0, sigma, n, alternative='two-sided'):
    """Z-test for population mean (known sigma).
    
    Returns: (z_stat, p_value)
    """
    se = sigma / np.sqrt(n)
    z = (sample_mean - mu_0) / se
    
    if alternative == 'two-sided':
        p = 2 * (1 - normal_cdf(abs(z)))
    elif alternative == 'greater':
        p = 1 - normal_cdf(z)
    else:  # 'less'
        p = normal_cdf(z)
    
    return z, p


def t_test_one_sample(data, mu_0, alternative='two-sided'):
    """One-sample t-test (unknown sigma).
    
    Returns: (t_stat, p_value, df)
    """
    data = np.asarray(data, dtype=float)
    n = len(data)
    mean = np.sum(data) / n
    s = np.sqrt(np.sum((data - mean) ** 2) / (n - 1))
    se = s / np.sqrt(n)
    t_stat = (mean - mu_0) / se
    df = n - 1
    
    if alternative == 'two-sided':
        p = 2 * (1 - stats.t.cdf(abs(t_stat), df))
    elif alternative == 'greater':
        p = 1 - stats.t.cdf(t_stat, df)
    else:
        p = stats.t.cdf(t_stat, df)
    
    return t_stat, p, df


# ── Example 1: Z-test for a fund's claimed return
# A fund claims average annual return of 12%. 
# We observe a sample of 36 years with mean 10.5%, known sigma = 8%.
z_stat, p_val = z_test(0.105, 0.12, 0.08, 36, 'two-sided')
alpha = 0.05
print("Z-Test: Is the fund's true mean return different from 12%?")
print(f"  H_0: mu = 12%,  H_1: mu != 12%")
print(f"  z = {z_stat:.4f}, p-value = {p_val:.4f}")
print(f"  Decision at alpha={alpha}: {'Reject H_0' if p_val < alpha else 'Fail to reject H_0'}")

# ── Example 2: T-test with sample data
# Monthly excess returns of a strategy
excess_returns = rng.normal(0.005, 0.03, 60)  # 60 months
t_stat, p_val, df = t_test_one_sample(excess_returns, 0.0, 'greater')
print(f"\nT-Test: Are excess returns significantly positive?")
print(f"  H_0: mu <= 0,  H_1: mu > 0")
print(f"  Sample mean = {arithmetic_mean(excess_returns):.4%}")
print(f"  t = {t_stat:.4f}, df = {df}, p-value = {p_val:.4f}")
print(f"  Decision at alpha={alpha}: {'Reject H_0' if p_val < alpha else 'Fail to reject H_0'}")

**Interpreting Example 1 (Z-test):**

The fund claims 12% average return, but our 36-year sample shows only 10.5%. Is this a meaningful difference, or just sampling noise?

The z-statistic of -1.125 means the sample mean is about 1.1 standard errors below the claimed mean. The p-value (about 0.26) tells us: if the true mean really is 12%, there's a 26% chance of seeing a sample mean as far from 12% as 10.5%. That's not unusual enough to reject the claim -- we fail to reject at the 5% level.

**Interpreting Example 2 (T-test):**

We're testing whether a trading strategy produces positive excess returns. The t-test accounts for the fact that we're estimating volatility from the sample (which adds uncertainty).

Now let's visualize how Type I and Type II errors work.**Interpreting Example 1 (Z-test):**

### Hypothesis testing — the framework

Hypothesis testing answers: "Is the observed result *meaningfully different* from what we'd expect by chance?"

The structure is always the same:

1. **Null hypothesis $H_0$:** The "default" claim (e.g., "the mean return is 8%")
2. **Alternative hypothesis $H_a$:** The claim we're investigating (e.g., "the mean return is *not* 8%")
3. **Test statistic:** A function of the data (e.g., the Z-statistic)
4. **Decision rule:** Reject $H_0$ if the test statistic is "extreme" enough
5. **P-value:** Probability of observing data this extreme if $H_0$ is true

### Z-test vs t-test

| Test | When to Use |
|------|-------------|
| Z-test | Population variance $\sigma^2$ is *known* |
| t-test | Population variance is *unknown* (estimated from sample) |

In practice, the population variance is almost never known, so the **t-test** is the workhorse. The Z-test appears mostly in textbooks and exam problems.

### One-sided vs two-sided tests

* **Two-sided** (default): Test whether the parameter differs from $H_0$ in *either* direction
* **One-sided** (greater or less): Test for differences in only one direction

One-sided tests have *higher power* (more likely to reject $H_0$ when it's false) but are appropriate only when you have strong prior reason to expect deviation in one direction.

> **Common Mistake:** Researchers sometimes choose the direction of the one-sided test *after* seeing the data. This is *invalid* — it inflates the false positive rate. The direction must be chosen *before* looking at the data.

In [ ]:
# ── Visualization: Type I and Type II errors
fig, ax = plt.subplots(figsize=(12, 6))

x = np.linspace(-4, 8, 500)
# H0: mu = 0
ax.plot(x, normal_pdf(x, 0, 1), color=PRIMARY, linewidth=2, label='H_0: mu=0')
# True distribution: mu = 2
ax.plot(x, normal_pdf(x, 2, 1), color=SECONDARY, linewidth=2, label='True: mu=2')

# Critical value for one-sided test at alpha=0.05
z_crit = 1.645
ax.axvline(z_crit, color='black', linestyle='--', linewidth=1.5, label=f'Critical z = {z_crit}')

# Shade Type I error (alpha)
x_alpha = np.linspace(z_crit, 4, 100)
ax.fill_between(x_alpha, normal_pdf(x_alpha, 0, 1), alpha=0.3, color=PRIMARY, label=f'Type I (alpha)')

# Shade Type II error (beta)
x_beta = np.linspace(-4, z_crit, 100)
ax.fill_between(x_beta, normal_pdf(x_beta, 2, 1), alpha=0.3, color=SECONDARY, label=f'Type II (beta)')

ax.set_xlabel('Test Statistic')
ax.set_ylabel('Density')
ax.set_title('Type I and Type II Errors in Hypothesis Testing')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Reading the error visualization:**

- The **blue curve** is the distribution under $H_0$ (no effect). The blue shaded area to the right of the critical value is **Type I error** ($\alpha$) -- the probability of incorrectly rejecting a true null.
- The **orange curve** is the true distribution (there is a real effect of size 2). The orange shaded area to the left of the critical value is **Type II error** ($\beta$) -- the probability of failing to detect a real effect.
- **Power** = 1 - $\beta$ = the white area under the orange curve to the right of the critical value.

Notice the trade-off: moving the critical value to the left reduces Type II errors (more power) but increases Type I errors (more false positives). You can't reduce both simultaneously unless you increase the sample size.**Reading the error visualization:**

### Type I and Type II errors

Hypothesis testing can fail in two ways:

| | $H_0$ True | $H_0$ False |
|---|-----------|-------------|
| **Reject $H_0$** | Type I error (false positive) | Correct (true positive) |
| **Fail to reject $H_0$** | Correct (true negative) | Type II error (false negative) |

* **Type I error rate** $\alpha$: probability of rejecting a true null. Set by the researcher (typically 5%).
* **Type II error rate** $\beta$: probability of failing to reject a false null. Determined by sample size, true effect size, and $\alpha$.
* **Power** = $1 - \beta$: probability of correctly rejecting a false null. Researchers want power $\geq 0.80$.

### The trade-off

Decreasing $\alpha$ (more conservative, fewer false positives) *increases* $\beta$ (more false negatives). The only way to reduce both simultaneously is to **collect more data** — larger samples shrink the standard errors and increase the test's resolving power.

### Why p-values are misunderstood

Common misconceptions:
1. **"P-value is the probability that $H_0$ is true."** No — it's the probability of observing data this extreme *given* that $H_0$ is true.
2. **"$p < 0.05$ means the result is important."** No — statistical significance ≠ economic significance. With huge samples, trivially small effects become "significant."
3. **"$p > 0.05$ means $H_0$ is true."** No — failing to reject $H_0$ is not the same as proving it. The test simply lacks evidence to reject.

> **CFA Exam Tip:** The CFA exam tests these concepts repeatedly. Be especially careful with the asymmetry: failure to reject $H_0$ does NOT prove $H_0$. The data is "consistent with $H_0$" but may also be consistent with many alternatives we didn't test. This is a critical philosophical point in inferential statistics.

---
## Summary: Key Statistical Measures for Investors

| Measure | Formula | Investment Interpretation |
|---|---|---|
| Arithmetic Mean | $\frac{1}{n}\sum x_i$ | Best single-period return forecast |
| Geometric Mean | $(\prod(1+x_i))^{1/n} - 1$ | Actual compound growth rate |
| Standard Deviation | $\sqrt{\frac{1}{n-1}\sum(x_i-\bar{x})^2}$ | Total risk of an investment |
| Sharpe Ratio | $(\bar{R}-R_f)/\sigma$ | Excess return per unit of risk |
| Skewness | Third standardized moment | Asymmetry of returns (crash risk) |
| Excess Kurtosis | Fourth moment - 3 | Tail fatness (extreme event risk) |

---
## References

1. **CFA Institute**, *CFA Program Curriculum Level I*, Quantitative Methods.
2. **DeFusco, R., McLeavey, D., Pinto, J., & Runkle, D.**, *Quantitative Investment Analysis*, 3rd ed., CFA Institute/Wiley, 2015.
3. **Wackerly, D., Mendenhall, W., & Scheaffer, R.**, *Mathematical Statistics with Applications*, 7th ed., Cengage, 2008.
4. **Casella, G. & Berger, R.**, *Statistical Inference*, 2nd ed., Cengage, 2002.

### Summary: the statistician's checklist

When analysing financial data, work through this checklist:

1. **Visualise first.** Always plot histograms, scatter plots, and time series before computing statistics.
2. **Choose the right central tendency.** Mean for symmetric data; median for skewed; geometric for compound returns.
3. **Use sample formulas with $n-1$.** Whenever you have a sample (not a population), apply Bessel's correction.
4. **Check for non-normality.** Compute skewness and kurtosis. Consider QQ-plots and formal tests (Jarque-Bera, Shapiro-Wilk).
5. **Don't assume independence.** Financial data has serial correlation, regime changes, and clustering. Standard errors that assume IID are too small.
6. **Interpret p-values cautiously.** Statistical significance is necessary but not sufficient for economic relevance.
7. **Report effect sizes, not just p-values.** A 0.1 percentage point return difference may be statistically significant but practically meaningless.

> **Key Concept:** Statistics is a *tool*, not a *substitute* for thinking. Mechanical application of formulas without understanding context, distribution, and economic meaning produces misleading results. The CFA curriculum emphasises judgment alongside computation — and this is exactly what professional practice requires.

### References

* **Bodie, Kane, Marcus** — *Investments*, foundational treatment of risk-return statistics.
* **Hull, J. C.** — *Options, Futures, and Other Derivatives*, especially Chapters on volatility estimation.
* **Casella, G., & Berger, R.** — *Statistical Inference*, the standard graduate text.
* **CFA Institute** — Quantitative Methods readings (Level 1).